# Addestramento e Valutazione Modello NER per Testi Storici
Questo notebook carica i dati generati da Claude (`train.jsonl`, `val.jsonl`, `test_gold.jsonl`), effettua il fine-tuning di un modello BERT Italiano, e calcola le metriche (F1-Score, Precision, Recall) usando seqeval.

**Istruzioni:**
1. Esegui la cella sottostante per installare le dipendenze.
2. Carica i file `.jsonl` generati nell'ambiente Colab.
3. Esegui tutte le celle sequenzialmente.

In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate

## 1. Caricamento Dataset e Setup

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

# Assicurati di aver caricato i file train.jsonl, val.jsonl e test_gold.jsonl nel file system di Colab
train_data = load_jsonl('train.jsonl')
val_data = load_jsonl('val.jsonl')
test_data = load_jsonl('test_gold.jsonl')

# Creiamo la label list unica
all_tags = set()
for data in [train_data, val_data, test_data]:
    for item in data:
        for tag in item['ner_tags']:
            all_tags.add(tag)

label_list = sorted(list(all_tags))
# Assicuriamoci che 'O' sia al primo posto
if 'O' in label_list:
    label_list.remove('O')
    label_list = ['O'] + label_list

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print("Label disponibili:", label_list)

def convert_to_dataset(data):
    dict_data = {"tokens": [], "ner_tags": []}
    for item in data:
        dict_data["tokens"].append(item["tokens"])
        # Convertiamo i tag testuali nei loro ID numerici
        dict_data["ner_tags"].append([label2id[t] for t in item["ner_tags"]])
    return Dataset.from_dict(dict_data)

datasets = DatasetDict({
    'train': convert_to_dataset(train_data),
    'validation': convert_to_dataset(val_data),
    'test': convert_to_dataset(test_data)
})

print(datasets)

## 2. Tokenizzazione (Allineamento Subwords)

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "dbmdz/bert-base-italian-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Ignora i token speciali (es. [CLS], [SEP])
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                # Per subword successive, impostiamo il tag a -100 per non penalizzare il modello
                # Oppure potremmo usare lo stesso tag, ma la prassi in PyTorch/HF è -100
                label_ids.append(-100) 
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

## 3. Metriche di Valutazione (Seqeval per calcolo F1-Score)

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    
    # Se vogliamo dare più peso alle classi rare, la Weighted-Loss andrà definita nel Trainer custom.
    # Per ora usiamo le metriche standard globali.
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 4. Addestramento del Modello

In [ ]:
from transformers import DataCollatorForTokenClassification, AutoModelForTokenClassification, TrainingArguments, Trainer

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Parametri classici per fine-tuning NER
training_args = TrainingArguments(
    output_dir="./bert-ner-storico",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Valutazione Definitiva (Test Gold Out-Of-Domain)
Qui dimostriamo che il modello è capace di generalizzare su testi antichi che non ha MAI visto.

In [ ]:
predictions, labels, metrics = trainer.predict(tokenized_datasets["test"])
print("\n--- RISULTATI TEST GOLD OUT-OF-DOMAIN ---")
print(f"F1 Score Assoluto: {metrics['test_f1']:.4f}")
print(f"Precisione: {metrics['test_precision']:.4f}")
print(f"Recall: {metrics['test_recall']:.4f}")

# Salvataggio finale per HuggingFace
trainer.save_model("bert-ner-storico-final")